# PCB 결함 검사 — 콜랩 Sanity Check

Team Convex | ATI 인턴 프로젝트

이 노트북은 새 콜랩 환경에서 repo 클론 → 의존성 설치 → 데이터 마운트 → **1-epoch 스모크 트레인** 까지
한 번에 검증하는 골격입니다.

> **실행 순서**: 셀을 위에서 아래로 순서대로 실행하세요.
> 
> **본격 학습**: `notebooks/pcb_train_colab.ipynb` 을 사용하세요.

## 셀 1 — Repo 클론 & 의존성 설치

In [ ]:
from google.colab import userdata
import subprocess, os

GITHUB_REPO = "june0922/pcb-defect-inspection"
PROJECT_DIR = "/content/pcb-defect-inspection"

try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = ""
    raise RuntimeError("화면 왼쪽 키 코 클릭 → GITHUB_TOKEN 등록 후 다시 실행")

if not os.path.exists(PROJECT_DIR):
    repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
    result = subprocess.run(
        ["git", "clone", repo_url, PROJECT_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(result.stderr.replace(GITHUB_TOKEN, "***"))
    print("✅ 클론 완료")
else:
    print("ℹ️  이미 클론됨")

%cd {PROJECT_DIR}
!pip install -q -r requirements.txt
print("설치 완료")

## 셀 2 — 데이터 준비

Google Drive 마운트 후 `dataset.zip` 에서 자동 압축 해제.

In [ ]:
from google.colab import drive
import os, shutil
from pathlib import Path

drive.mount('/content/drive')

DRIVE_ZIP = "/content/drive/MyDrive/pcb-defect-inspection/dataset.zip"
LOCAL_ZIP = Path("/content/pcb-defect-inspection/dataset.zip")

if not LOCAL_ZIP.exists() and os.path.exists(DRIVE_ZIP):
    print("Drive에서 dataset.zip 복사 중...")
    shutil.copy2(DRIVE_ZIP, LOCAL_ZIP)
    print("✅ 복사 완료")

raw_data = Path("/content/pcb-defect-inspection/dataset/PCBData")
assert LOCAL_ZIP.exists() or raw_data.exists(), (
    f"dataset.zip 또는 {raw_data}가 필요합니다.\n"
    "Drive에 dataset.zip을 업로드하거나 경로를 확인하세요."
)
print("✅ 데이터 준비 확인 완료")

## 셀 3 — config.yaml 콜랩 환경으로 전환 & 전처리

In [ ]:
import yaml

# env 를 colab 으로 전환
with open("config.yaml", "r") as f:
    cfg = yaml.safe_load(f)
cfg["env"] = "colab"
with open("config.yaml", "w") as f:
    yaml.dump(cfg, f)
print("env:", cfg["env"])

# 전처리 실행
!python src/preprocess.py --config config.yaml

## 셀 4 — 스모크 트레인 (epochs=1)

In [ ]:
import yaml

# epochs 를 1로 임시 변경
with open("config.yaml", "r") as f:
    cfg = yaml.safe_load(f)
cfg["train"]["epochs"] = 1
with open("config.yaml", "w") as f:
    yaml.dump(cfg, f)

!python src/train.py --config config.yaml

## 셀 5 — 판정 레이어 동작 확인 (inspect)

In [ ]:
import sys
sys.path.insert(0, "src")

from ultralytics import YOLO
from utils import load_config, get_paths
from pcb_inspect import inspect_image  # ✔️ 수정: inspect(표준 라이브러리)가 아닌 pcb_inspect

cfg = load_config("config.yaml")
paths = get_paths(cfg)

weight = paths["weights"] / "best.pt"
assert weight.exists(), f"best.pt 가 없습니다: {weight}"
model = YOLO(str(weight))

# TODO: 테스트할 샘플 이미지 경로로 변경
SAMPLE_IMAGE = str(paths["processed"] / "images" / "test" / "SAMPLE.jpg")  # TODO

result = inspect_image(SAMPLE_IMAGE, model, cfg)
print("판정:", result["verdict"])
print("결함 수:", result["defect_count"])
print("클래스별:", result["by_class"])
if result["review"]:
    print("REVIEW 대상:", result["review"])